In [1]:
import sys
sys.path.append("../")

from models.models import train_model_on_named_experiment

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


## Training Models:

#### Implemented Models:
| Model Class | Model Description | 
|------------|--------------------|  
| `AE` | Autoencoder | 
| `AEC`| Autoencoder Classifier |
| `scMEDAL-FE` | Domain Adversarial Autoencoder |
| `scMEDAL-FEC`| Domain Adversarial Autoencoder Classifier |
| `scMEDAL-RE`| Domain Enhancing Autoencoder Classifier | 

#### Named Experiments:
| Valid Named Experiment | Dataset |  n_clusters | n_pred |
|------------------------|---------|-------------|--------|
| `AML`| Acute Myeloid Leukemia | 19 | 21 |
| `ASD`| Autism Spectrum Disorder | 31 | 17 | 
| `HH` | Healthy Heart | 147 | 13 | 

**Note:** If training on other datasets, the configs will need to be passed in as dictionaries to `model_kwargs` and `train_kwargs`.

`quick` is a boolean flag that can be passed to `train_kwargs` which shortens training to only 1 fold for 3 epochs.

In [ ]:
ae_aml = train_model_on_named_experiment("AE", "AML", model_kwargs={"n_latent_dims":50}, train_kwargs={
        "quick": True,                         # let us control epochs ourselves
        "quick_epochs":10,                      # 10 epochs
        "quick_folds":[2],
        
    })

In [ ]:
scmedalre_aml = train_model_on_named_experiment("scMEDAL-RE", "AML", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True,
        "quick_epochs":10,                      # 10 epochs
        "quick_folds":[2],})

In [ ]:
scmedalfe_aml = train_model_on_named_experiment("scMEDAL-FE", "AML", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True,
        "quick_epochs":10,                      # 10 epochs
        "quick_folds":[2],})

In [ ]:
# scmedalre_asd = train_model_on_named_experiment("scMEDAL-RE", "ASD", model_kwargs={"n_latent_dims":50}, train_kwargs={"quick":True})

#             self.training_configs._replace(epochs=3)
#             self.model_params['epochs'] = 3

## Analyzing Models:

In [2]:
import analysis.analysis as aa

analysis_name = "AML_default"
model_folder_dict = {
    "ae":"run_crossval_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-ae_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sample_size-10000_n_components-50_get_cf_batch-False_2025-07-01_13-31",
    "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-01_18-18",
    # "aec":"run_crossval_n_latent_dims-2_layer_units-512-132_n_pred-21_use_batch_norm-True_scaling-min_max_model_type-aec_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sample_size-10000_2025-06-26_20-21",
    # "scmedalfe":"run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-2_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-06-26_20-42",
    # "scmedalfec":"run_crossval_loss_gen_weight-1_loss_recon_weight-2000_loss_class_weight-1_n_latent_dims-2_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfec_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-06-26_20-53",
     "scmedalre":"run_crossval_loss_recon_weight-110.0_loss_latent_cluster_weight-0.1_n_latent_dims-50_layer_units-512-132_kl_weight-0.0_scaling-min_max_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-01_13-32",
}


aml_analysis = aa.AMLAnalysis(model_folder_dict, analysis_name)

In [3]:
aml_analysis.clustering_scores(model_folder_dict)

Directory created or already exists: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/outputs/AML/compare_models/log_transformed_2916hvggenes/AML_default
ae /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/outputs/AML/latent_space/log_transformed_2916hvggenes/ae/run_crossval_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-ae_batch_size-512_epochs-500_patience-30_compute_latents_callback-False_sample_size-10000_n_components-50_get_cf_batch-False_2025-07-01_13-31
scmedalfe /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/outputs/AML/latent_space/log_transformed_2916hvggenes/scmedalfe/run_crossval_loss_gen_weight-1_loss_recon_weight-4000_loss_class_weight-1_n_latent_dims-50_layer_units-512-132_use_batch_norm-True_scaling-min_max_model_type-scmedalfe_batch_size-512_epochs-500_patience-30_sample_size-10000_2025-07-01_18-18

In [3]:
batches = ["AML420B", "BM5", "MUTZ3"]
celltype = ["Mono", "Mono-like"]
# for this genomap config, you need to compute first the reconstructions from scMEDAL-FE and scMEDAL-RE
gfd = aml_analysis.genomap(model_folder_dict,
                            n_batches = 19,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "Cell",
                            gene_index_col = "Gene",
                            celltype = ["Mono", "Mono-like"],
                            batches=batches,
                            models=['scmedalre'],
                            types=["train"], 
                            splits=[2],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            min_val = -1,
                            max_val =2)

Computing genomap for ['train'] [2] ['scmedalre']
train 2 scmedalre
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/data/AML_data/log_transformed_2916hvggenes/splits/split_2/train
batch_sel ['AML420B', 'BM5', 'MUTZ3']
Selected 1 cell from batch AML420B with celltype Mono.
Selected 1 cell from batch AML420B with celltype Mono-like.
Selected 1 cell from batch BM5 with celltype Mono.
No cells available for celltype ['Mono', 'Mono-like'] in batch BM5. Skipping batch.
No cells available for celltype ['Mono', 'Mono-like'] in batch MUTZ3. Skipping batch.
Selected 1 cell from batch MUTZ3 with celltype Mono-like.
Selected 296 additional cells from specified celltypes.
input_train
Reading data from: /archive/bioinformatics/DLLab/AixaAndrade/src/gitfront/scMEDAL_for_scRNAseq/Experiments/data/AML_data/log_transformed_2916hvggenes/splits/split_2/train
fe_ae_recon_train
recon_train
recon_batch_train_BM3
recon_batch_train_AML329
recon_batch_t

/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


batches 2 select from ['AML420B', 'BM5', 'MUTZ3']
AML420B-D35_TGAATTCTTCAC comes from batch AML420B
AML420B-D0_GGCGCTTCCGCN comes from batch AML420B
BM5-34p38n_TGTGGACAAGCA comes from batch BM5
MUTZ3_CGAGTCGCAATG comes from batch MUTZ3
original batch list: ['AML420B', 'AML420B', 'BM5', 'MUTZ3']
cell_ids: ['AML420B-D35_TGAATTCTTCAC', 'AML420B-D0_GGCGCTTCCGCN', 'BM5-34p38n_TGTGGACAAGCA', 'MUTZ3_CGAGTCGCAATG']
Reading data from: /endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/outputs/AML/compare_models/log_transformed_2916hvggenes/AML_default/CMmultibatch_300_cells_per_batch_19batches_Mono_Mono-like_with_2fe_input


/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/archive/bioinformatics/DLLab/shared/CondaEnvironments/scMEDAL/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Input does not contain NaNs
subset_input_z_scores shape: (6300, 2916)


...

/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])
/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])


Shape of std_across_batches: (54, 54)


/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])
/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])


Shape of std_across_batches: (54, 54)


/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])
/endosome/archive/bioinformatics/DLLab/src/AixaAndrade/gitfront/dev/scMEDAL_for_scRNAseq/demo/../utils/genomaps_utils.py:996: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 0.9, 0.97])


In [ ]:
# I think this should work for asd but I would appreciate if you could guide me with this, because I think the ASD analysis is not implemented,
# the GenomapPipeline in genomap and plot should work for all datasets, ideally I would push it to genomap_utils

dict_batches = {
    "donor_5531": "ASD",
    "donor_5945": "ASD",
    "donor_5419": "ASD",
    "donor_6032": "control",
    "donor_5242": "control",
    "donor_5976": "control",
}
print("dict_batches:", dict_batches)

batches = [
    "donor_5531",
    "donor_5945",
    "donor_5419",
    "donor_6032",
    "donor_5242",
    "donor_5976",
]

gfd = asd_analysis.genomap(model_folder_dict, #change for asd runs (scmedalfe and scmedalre)
                            n_batches = 31,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "cell",
                            gene_index_col = "gene_ids",
                            celltype = ["L2/3"],
                            batches=batches,
                            models=['scmedalre'],
                            types=["train"], 
                            splits=[2],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            n_cells_2_plot = 6,
                            min_val = -2,
                            max_val =6)

In [ ]:
# I think this should work for HH but I would appreciate if you could guide me with this, because I think theHH analysis is not implemented,
# the GenomapPipeline in genomap and plot should work for all datasets, ideally I would push it to genomap_utils



batches = ["H0037_Apex", "HCAHeart7836681", "HCAHeart8102861", "H0015_septum"]
gfd = hh_analysis.genomap(model_folder_dict, #change for asd runs (scmedalfe and scmedalre)
                            n_batches = 147,
                            num_iter = 2, # for quick test, otherwise 100
                            cell_id_col = "_index",
                            gene_index_col = "_index",
                            celltype = ["Ventricular_Cardiomyocyte", "Endothelial", "Fibroblast", "Pericytes"],
                            batches=batches,
                            models=['scmedalre'],
                            types=["train"], 
                            splits=[2],
                            add_inputs_fe= True,
                            extra_recon = "fe",
                            n_cells_2_plot = 4,
                            min_val = -2,
                            max_val =8)

In [ ]:
#because it's a quick test, I am only able to get split 1
processors = aml_analysis.umap(model_folder_dict, types=["train"], splits=[2])
processors